# Anthropic Claude API 구조 실습

A-PI 사전조사 — 7개 항목을 `apicompare.providers.anthropic_adapter.AnthropicAdapter` 로 실측하고 `evidence/anthropic/*.json` 을 남긴다.

- SDK: `anthropic` 1.x (Messages API). 레거시 Text Completions 는 1.x 에서 제거됨.
- 기본 모델: `claude-haiku-5-5` ($0.10 / $0.50 per MTok). 모든 유료 셀은 `max_tokens <= 64`.
- **현재 `.env` 에 `ANTHROPIC_API_KEY` 가 없으면 모든 섹션이 `[BLOCKED: no key]` 로 끝난다(에러 없이).** 키를 넣고 다시 실행하면 LIVE 경로가 돈다.
- 규칙: 키·조직 ID·이메일을 출력하지 않는다. SDK 클라이언트 객체나 환경변수 전체를 print 하지 않는다. 응답 헤더는 어댑터의 화이트리스트(`anthropic-ratelimit-*`, `request-id` 등)만 저장.

| § | 항목 | 어댑터 메서드 | 과금 |
|---|---|---|---|
| 1 | 모델 목록 조회 | `list_models()` | 무료(키만 있으면 크레딧 0 이어도 됨) |
| 2 | 모델 호출 | `generate()` | 유료(≤64 토큰) |
| 3 | Token Usage | `generate().usage` + `count_tokens()` | §2 결과 재사용 + count_tokens 무료 |
| 4 | Usage 조회 | `usage_report()` (raw HTTP) | Admin 키 필수 |
| 5 | 비용 확인 | `cost()` 로컬 + `cost_report()` | Admin 키 필수(cost_report) |
| 6 | 상태 및 오류 확인 | `health()` + 의도적 오류 | 무료(health) / 소액 |
| 7 | 호출 예제 | SDK 직접 호출 + curl | 유료(≤64 토큰) |

## 0. 준비

- `.env` 는 `apicompare.config.load_env()` 로만 읽는다(키 값은 어댑터 내부에서만 사용).
- `uv run jupyter lab` 으로 열어야 Python 3.12 + anthropic 1.x 커널이 잡힌다(시스템 Python 3.9 는 0.x SDK 가 설치돼 실습이 꼬인다).
- 어댑터는 `timeout_s=30`, `max_retries=0` 으로 고정 — SDK 기본(600초, 2회 재시도)과 다르다. 3사 latency 비교를 왜곡하지 않기 위한 설정.

In [ ]:
import sys
assert sys.version_info >= (3, 12), "Python 3.12+ 가 필요합니다. 저장소 루트에서 `uv run jupyter lab` 으로 실행하세요."

import anthropic
assert anthropic.__version__.startswith("1."), f"anthropic 1.x 필요 (현재 {anthropic.__version__})"
print("python", sys.version.split()[0], "| anthropic SDK", anthropic.__version__)

from apicompare.config import load_env, keys
load_env()  # python-dotenv 직접 호출 대신 프로젝트 공통 로더

from apicompare.providers import get_adapter
from apicompare.evidence import save_evidence, summary
from apicompare.models import to_dict, NotSupported

adapter = get_adapter("anthropic", timeout_s=30, max_retries=0)
MODEL = adapter.default_model          # claude-haiku-5-5
LIVE = adapter.mode == "LIVE"
HAS_ADMIN = bool(adapter.admin_key)
PROMPT = "대한민국의 수도를 한 문장으로 알려줘."

print(f"mode = [{adapter.mode}]  model = {MODEL}  admin_key = {HAS_ADMIN}")
print("keys(존재 여부만):", keys())


def blocked(item: str, note: str = "no key"):
    # NO_KEY 공통 처리: BLOCKED 출력 + blocked evidence 저장 (키 값은 어디에도 남지 않는다)
    print(f"[BLOCKED: {note}]")
    return save_evidence("anthropic", item, {"blocked": note, "error": to_dict(adapter.no_key_error())}, "blocked", note=note, model=MODEL)

## 1. 모델 목록 조회

- 공식 방법: `GET https://api.anthropic.com/v1/models` (목록) / `GET /v1/models/{model_id}` (단일, alias → 실제 ID 해석). 헤더 `x-api-key`, `anthropic-version: 2023-06-01`.
- SDK: `client.models.list()` — 페이지 객체를 순회하면 `after_id` 커서로 자동 페이지네이션(`limit` 은 페이지 크기, `.data` 를 쓰지 말 것).
- 응답 필드: `id`, `display_name`, `created_at`, `lifecycle(active|deprecated|retired)`, `deprecated_at`, `retires_at`, `line`, `max_input_tokens`(컨텍스트), `max_tokens`(최대 출력), `capabilities{thinking, effort, image_input, pdf_input, structured_outputs, batch, ...}`. **가격 필드는 없다.**
- 주의: 일반 API 키로 호출 가능하고 **토큰 과금이 없다 → 키만 있으면 크레딧이 0 이어도 LIVE 로 실측 가능**. 조직이 쓸 수 없는 모델은 목록에 안 나오고, 호출 시 존재하지 않는 모델과 똑같이 `404 not_found_error` 를 준다. 현행 모델 ID 는 날짜 접미사 없는 고정 스냅샷(`claude-haiku-5-5` 등).
- 출처: https://platform.claude.com/docs/en/api/models/list.md · https://platform.claude.com/docs/en/about-claude/models/overview.md

In [ ]:
if LIVE:
    models = adapter.list_models()
    print(f"{len(models)} models (lifecycle 미지정 → active + deprecated)")
    print(f"{'id':30} {'display_name':24} {'ctx':>9} {'max_out':>8} lifecycle retires_at")
    for m in models[:20]:
        print(f"{m.id:30} {str(m.display_name):24} {str(m.context_window):>9} {str(m.max_output_tokens):>8} {m.lifecycle} {m.shutdown_date}")
else:
    models = []
    print("[BLOCKED: no key]  (키만 있으면 크레딧 없이도 이 항목은 LIVE 가능 — GET /v1/models 는 무료)")

In [ ]:
# 원본 객체/필드 확인: capabilities 는 untyped dict → bracket 접근
if LIVE and models:
    m0 = next((m for m in models if m.id == MODEL), models[0])
    caps = m0.capabilities
    print(m0.id, "| raw keys:", sorted(m0.raw.keys()))
    print("thinking.adaptive :", caps.get("thinking", {}).get("types", {}).get("adaptive"))
    print("effort            :", {k: v.get("supported") for k, v in caps.get("effort", {}).items() if isinstance(v, dict)})
    print("structured_outputs:", caps.get("structured_outputs"), "| image_input:", caps.get("image_input"))
else:
    print("[BLOCKED: no key]")

관찰 메모: (실행 후 채우기) — 모델 수, 기본 모델의 max_input_tokens / max_tokens, retires_at 이 채워진 모델이 있는지.

In [ ]:
if LIVE and models:
    path = save_evidence("anthropic", "01_models", {"count": len(models), "models": [to_dict(m) for m in models]}, "live", model=MODEL)
    print("saved:", path)
else:
    blocked("01_models")

## 2. 모델 호출

- 공식 방법: `POST https://api.anthropic.com/v1/messages` 하나(스트리밍은 같은 URL 에 `stream: true`, 배치는 `/v1/messages/batches`). 필수 바디: `model`, `max_tokens`, `messages[{role: user|assistant, content}]`. 선택: `system`, `thinking`, `output_config{effort, format}`, `tools`, `stop_sequences`, `service_tier`, `cache_control`.
- SDK: `client.messages.create(...)` / 헤더까지 보려면 `client.messages.with_raw_response.create(...)` → `.headers`, `.parse()`.
- 응답 `content` 는 **블록 배열**(`text` / `thinking` / `tool_use` ...). `block.type == "text"` 를 확인하고 `.text` 를 읽어야 한다(현행 모델은 thinking 이 기본 ON 이라 thinking 블록이 앞에 올 수 있음).
- `stop_reason`: `end_turn | max_tokens | stop_sequence | tool_use | pause_turn | refusal | model_context_window_exceeded`. `refusal` 이면 HTTP 200 이고 `stop_details{category, explanation}` 가 붙는다 → 어댑터는 `status="ok"`, `finish_reason="refusal"` 로 그대로 기록한다(오류가 아니라 '응답'이다).
- 주의: 현행 모델(Haiku 5.5 / Sonnet 5.5 / Opus 5.5 / Fable 5.1)은 `temperature / top_p / top_k` 를 받지 않는다(서버 400). SDK 1.x 는 파라미터 자체를 제거해 그냥 넘기면 **네트워크 이전 TypeError**. 어댑터는 세 파라미터를 제거하고 `raw["dropped_params"]` 에 기록한다. 구조화 출력은 `output_config.format = {"type": "json_schema", "schema": {...}}` (어댑터 `json_schema=`).
- 출처: https://platform.claude.com/docs/en/api/messages/create.md · https://platform.claude.com/docs/en/build-with-claude/working-with-messages.md

In [ ]:
if LIVE:
    result = adapter.generate(PROMPT, model=MODEL, max_output_tokens=64)   # 유료: 입력 ~30 + 출력 ≤64 토큰
    print("status      :", result.status)
    print("text        :", result.text)
    print("finish_reason:", result.finish_reason, "| latency_ms:", result.latency_ms, "| request_id:", result.request_id)
    if result.error:
        print("error       :", to_dict(result.error))
else:
    result = adapter.generate(PROMPT)   # NO_KEY → 네트워크 없이 error(AUTH) CallResult
    print("[BLOCKED: no key]", "| status:", result.status, "| error.category:", result.error.category)

In [ ]:
# 원본 객체/필드 확인: content 블록 타입, stop_reason/stop_details, 화이트리스트 헤더
if LIVE and result.status == "ok":
    print("content_block_types:", result.raw.get("content_block_types"))
    print("stop_reason        :", result.raw.get("stop_reason"), "| stop_details:", result.raw.get("stop_details"))
    if result.finish_reason == "refusal":
        print("→ 안전 분류기 거절: HTTP 200 이지만 본문이 비어 있을 수 있음. stop_details.category 확인.")
    print("response_headers   :", result.response_headers)   # anthropic-ratelimit-*, request-id 만
    print("dropped_params     :", result.raw.get("dropped_params"))
else:
    print("[BLOCKED: no key]")

관찰 메모: (실행 후 채우기) — 응답 텍스트, stop_reason, latency_ms, content 블록에 thinking 이 포함됐는지.

In [ ]:
if LIVE and result.status == "ok":
    print("saved:", save_evidence("anthropic", "02_call", result, "live", model=MODEL))
elif LIVE:
    print("saved:", save_evidence("anthropic", "02_call", result, "blocked", note=result.error.category if result.error else "error", model=MODEL))
else:
    blocked("02_call")

## 3. Token Usage

- 공식 방법: 모든 Messages 응답의 `usage` 객체. 사전 계산은 `POST /v1/messages/count_tokens` (무료, 별도 RPM; 출력 토큰은 사전 계산 불가, 모델별 토크나이저라 `model` 필수).
- 필드: `input_tokens`(마지막 캐시 브레이크포인트 이후 **비캐시** 입력), `cache_creation_input_tokens`, `cache_read_input_tokens`, `cache_creation{ephemeral_5m_input_tokens, ephemeral_1h_input_tokens}`, `output_tokens`(**thinking 포함, 과금 총합**), `output_tokens_details.thinking_tokens`(분해값), `service_tier`, `speed`, `inference_geo`.
- **`total_tokens` 필드가 없다.** 어댑터의 `UsageNormalized` 는 아래 식으로 계산하고 `total_is_computed=True` 로 표시한다.

| 정규화 필드 | Anthropic 산출식 |
|---|---|
| `input_tokens` | `input_tokens + cache_creation_input_tokens + cache_read_input_tokens` |
| `cached_input_tokens` | `cache_read_input_tokens` |
| `cache_write_tokens` | `cache_creation_input_tokens` |
| `output_tokens` | `output_tokens` (thinking 포함) |
| `reasoning_tokens` | `output_tokens_details.thinking_tokens` (output 에 이미 포함 — 더하지 않음) |
| `total_tokens` | `input_tokens(정규화) + output_tokens` |

- 주의: Claude 4.7+ 토크나이저는 동일 텍스트에 약 30% 많은 토큰 → 타사와 '토큰 수' 직접 비교는 불공정, 비용($) 으로도 비교. `count_tokens` 는 캐시 로직 미적용 추정치.
- 출처: https://platform.claude.com/docs/en/api/messages/create.md · https://platform.claude.com/docs/en/build-with-claude/token-counting.md · https://platform.claude.com/docs/en/build-with-claude/prompt-caching.md

In [ ]:
if LIVE and result.status == "ok" and result.usage:
    u = result.usage
    pre = adapter.count_tokens(PROMPT, model=MODEL)   # 무료
    print(f"count_tokens(사전) = {pre}  |  응답 input_tokens(정규화, 캐시 포함) = {u.input_tokens}")
    print(f"input={u.input_tokens} (cached={u.cached_input_tokens}, cache_write={u.cache_write_tokens})  output={u.output_tokens} (thinking={u.reasoning_tokens})")
    print(f"total={u.total_tokens}  total_is_computed={u.total_is_computed}")
else:
    pre = None
    print("[BLOCKED: no key]")

In [ ]:
# 원본 usage 필드 (어댑터 raw) — total_tokens 키가 없음을 확인
if LIVE and result.status == "ok" and result.usage:
    import json
    print(json.dumps(result.usage.raw, ensure_ascii=False, indent=2))
    print("'total_tokens' in raw:", "total_tokens" in result.usage.raw)
else:
    print("[BLOCKED: no key]")

관찰 메모: (실행 후 채우기) — count_tokens 와 응답 input_tokens 차이, thinking_tokens 가 output 의 몇 %인지, output_tokens_details 가 beta 헤더 없이도 채워지는지(open question).

In [ ]:
if LIVE and result.status == "ok" and result.usage:
    payload = {"usage": to_dict(result.usage), "count_tokens_pre_call": to_dict(pre), "raw_usage": result.usage.raw}
    print("saved:", save_evidence("anthropic", "03_usage", payload, "live", model=MODEL))
else:
    blocked("03_usage", "no key" if not LIVE else "call failed")

## 4. Usage 조회

- 공식 방법: **Usage & Cost Admin API** `GET https://api.anthropic.com/v1/organizations/usage_report/messages?starting_at=...&ending_at=...&bucket_width=1d&group_by[]=model`. 헤더 `x-api-key: $ANTHROPIC_ADMIN_KEY`, `anthropic-version: 2023-06-01`.
- 인증: **Admin API key(접두어 `sk-ant…admin`) + 조직(Organization) 계정 필수.** 개인 계정은 불가 → Console 에서 조직을 만들고 admin 역할로 Admin 키 발급. 일반 workspace 키는 401/403. Admin 키로 Messages API 호출은 불가(역방향도 마찬가지).
- SDK 에 이 엔드포인트 메서드가 **없다**(curl-only 문서) → `requests` 로 raw HTTP. 어댑터 `usage_report()` 가 이 호출을 감싼다.
- 응답: `data[]{starting_at, ending_at, results[]{uncached_input_tokens, cache_creation{ephemeral_5m_input_tokens, ephemeral_1h_input_tokens}, cache_read_input_tokens, output_tokens, model, workspace_id, api_key_id, service_tier, ...}}`, `has_more`, `next_page`. 입력 필드명이 Messages 응답(`input_tokens`)과 달리 `uncached_input_tokens` 다. 버킷 한도 1m 1,440 / 1h 168 / 1d 31. 데이터 지연 약 5분.
- **혼동 경고:** `client.beta.organization.analytics.usage_report / cost_report` 는 `/v1/organizations/analytics/*` — **Claude Enterprise(claude.ai 조직) 전용 Analytics API** 로 Analytics 키가 필요하고 Console 조직의 usage_report/messages 와 다른 API 다. SDK 에 보이는 메서드라고 이걸 쓰면 안 된다.
- 대안: platform.claude.com/usage 대시보드, 또는 호출마다 응답 usage 를 자체 적재해 집계(`apicompare.usagelog.aggregate()`).
- 출처: https://platform.claude.com/docs/en/manage-claude/usage-cost-api.md · https://platform.claude.com/docs/en/api/beta/organization/usage_report/retrieve_messages.md · https://platform.claude.com/docs/en/manage-claude/admin-api.md

In [ ]:
from datetime import datetime, timedelta, timezone
from apicompare import usagelog

now = datetime.now(timezone.utc)
start, end = now - timedelta(days=7), now

# 어댑터 내부의 raw HTTP (참고용 — SDK 메서드 없음):
#   requests.get("https://api.anthropic.com/v1/organizations/usage_report/messages",
#                params=[("starting_at", "2026-10-12T00:00:00Z"), ("ending_at", "2026-10-19T00:00:00Z"),
#                        ("bucket_width", "1d"), ("group_by[]", "model")],
#                headers={"x-api-key": "$ANTHROPIC_ADMIN_KEY", "anthropic-version": "2023-06-01"})
rep = adapter.usage_report(start, end, bucket="1d", group_by=("model",))
if isinstance(rep, NotSupported):
    print(f"[BLOCKED: {'no key' if not LIVE and not HAS_ADMIN else 'no admin key'}]")
    print("reason     :", rep.reason)
    print("alternative:", rep.alternative)
else:
    print(rep.source, rep.note)
    for b in rep.buckets[:10]:
        print(b.start, b.group, "in=", b.input_tokens, "out=", b.output_tokens)

In [ ]:
# 원본 확인 + Admin 키 유효성(GET /v1/organizations/me) + 자체 집계 대안
if not isinstance(rep, NotSupported) and rep.buckets:
    print("raw result[0]:", rep.buckets[0].raw)
else:
    print("[BLOCKED: no admin key]" if not HAS_ADMIN else "[no data in range]")
print("verify_admin_key:", to_dict(adapter.verify_admin_key()))   # 조직 ID 는 돌려주지 않는다
local = usagelog.aggregate(start, end)
print("usagelog(자체 집계):", local.note, "| buckets:", len(local.buckets))

관찰 메모: (실행 후 채우기) — Admin 키 발급 가능 여부(개인 계정이면 조직 생성 필요), uncached_input_tokens 와 §3 의 input_tokens 관계, 지연 시간.

In [ ]:
if isinstance(rep, NotSupported):
    note = "no key" if not LIVE and not HAS_ADMIN else "no admin key"
    print("saved:", save_evidence("anthropic", "04_usage_api", {"not_supported": to_dict(rep), "usagelog": to_dict(local)}, "blocked", note=note, model=MODEL))
    print(f"[BLOCKED: {note}]")
else:
    print("saved:", save_evidence("anthropic", "04_usage_api", {"report": to_dict(rep), "usagelog": to_dict(local)}, "live", model=MODEL))

## 5. 비용 확인

- 단가는 **어떤 API 로도 제공되지 않는다**(Models API 에 가격 필드 없음) → 공식 pricing 페이지를 날짜·출처와 함께 로컬 테이블(`apicompare.pricing.PRICE_TABLE`)로 관리하고 `usage × 단가` 로 계산한다(`adapter.cost()`).
- 사후 실비: `GET /v1/organizations/cost_report?starting_at=...&group_by[]=description` (Admin 키, SDK 미지원, **1d 버킷만**). `results[].amount` 는 **센트 단위 decimal 문자열**("123.45" = $1.2345) → `Decimal(amount)/100`. `model/token_type/cost_type` 은 `group_by[]=description` 을 줘야 채워진다. 1회 호출의 실제 청구액을 돌려주는 필드는 없다 → 호출 단위 비용은 반드시 로컬 계산.
- Haiku 5.5 단가(2026-10-18): 입력 $0.10 / 출력 $0.50 / 5m 캐시 쓰기 $0.125 / 캐시 읽기 $0.01 per MTok, **프롬프트(캐시 포함 총입력) 100K 초과 시 $0.50 / $2.50**. thinking 토큰은 output 단가. Batch 50% 할인, 1h 캐시 쓰기 2x, inference_geo us 1.1x.
- 출처: https://platform.claude.com/docs/en/about-claude/pricing.md · https://platform.claude.com/docs/en/api/beta/organization/cost_report/retrieve.md

In [ ]:
from apicompare import pricing

price = pricing.get_price("anthropic", MODEL)
print(f"단가표({price.as_of}, {price.source}): input ${price.input} / output ${price.output} / cache_write_5m ${price.cache_write_5m} / cache_read ${price.cached_input} per MTok")
print("note:", price.note, "| long_context:", price.long_context)

if LIVE and result.status == "ok" and result.usage:
    cost = adapter.cost(MODEL, result.usage, service_tier=result.service_tier)
    print("local estimate:", to_dict(cost))
    usagelog.append(result, cost if not isinstance(cost, NotSupported) else None, run_id="nb20")
else:
    cost = None
    print("[BLOCKED: no key]  (usage 가 없어 호출 단위 비용을 계산할 수 없음)")

In [ ]:
# 사후 실비 조회 (Admin 키 필요) — 원본 amount 는 센트 decimal 문자열
crep = adapter.cost_report(start, end)
if isinstance(crep, NotSupported):
    print(f"[BLOCKED: {'no key' if not LIVE and not HAS_ADMIN else 'no admin key'}]", "|", crep.reason)
else:
    print(crep.note)
    for b in crep.buckets[:10]:
        print(b.start, b.group.get("description"), "amount(cents)=", b.raw.get("amount"), "→ usd=", b.cost_usd)

관찰 메모: (실행 후 채우기) — 1회 호출 예상 비용(USD), cost_report 합계와 로컬 추정의 차이(지연 5분 감안).

In [ ]:
if cost is not None and not isinstance(cost, NotSupported):
    payload = {"local_estimate": to_dict(cost), "cost_report": to_dict(crep), "price": to_dict(price)}
    print("saved:", save_evidence("anthropic", "05_costs", payload, "live", model=MODEL))
else:
    blocked("05_costs", "no key" if not LIVE else "no call")

## 6. 상태 및 오류 확인

- 전용 `/health` 엔드포인트는 없다. 가장 저렴한 확인은 `GET /v1/models?limit=1` (토큰 과금 0): 200 = 네트워크+인증 OK, 401 = 키 문제, 403 = 권한. 어댑터 `health()` 가 이걸 호출하고 `anthropic-ratelimit-*` 헤더를 돌려준다. 서비스 장애 공지: status.anthropic.com.
- 오류 본문은 항상 `{"type":"error","error":{"type":"<error_type>","message":"...","details":{"error_code":"..."}?},"request_id":"req_..."}`.

| HTTP | error.type | 비고 |
|---|---|---|
| 400 | `invalid_request_error` | 잘못된 파라미터, 미지원 파라미터(temperature 등), **사용자 설정 지출 한도**('...usage limits'), **크레딧 소진**('credit balance is too low') |
| 401 | `authentication_error` | 키 누락/오류/만료 |
| 403 | `permission_error` | 권한/지역 제한 |
| 404 | `not_found_error` | 모델·엔드포인트 없음 **또는 조직 미허용 모델**(구분 안 됨) |
| 413 | `request_too_large` | 32MB 초과 |
| 429 | `rate_limit_error` | RPM/ITPM/OTPM(retry-after 있음) **또는 티어 월 지출 한도**(`details.error_code=enforced_spend_limit_reached`, retry-after 없음) |
| 500 | `api_error` | 내부 오류 |
| 504 | `timeout_error` | 처리 타임아웃 |
| **529** | `overloaded_error` | 과부하 — **503 이 아니다** |

- **(c) 529 분기:** SDK 의 `OverloadedError(529)` 는 `InternalServerError(>=500)` 의 하위 클래스가 **아니다**. `except anthropic.InternalServerError` 만 쓰면 529 를 놓친다 → `except anthropic.APIStatusError as e` 로 받아 `e.status_code` / `e.type` 으로 분기한다(어댑터 `error_from_exception()`). 순서: `APITimeoutError ⊂ APIConnectionError`(APIStatusError 의 형제) 를 먼저.
- SDK 기본은 `max_retries=2`, `timeout=600s` → 어댑터는 0 / 30s 로 고정. 비스트리밍 요청은 `max_tokens > 21,333` 이면 SDK 가 요청 전에 `ValueError("Streaming is required ...")`.

**(d) Rate limit 응답 헤더** (매 응답에 포함, 어댑터 화이트리스트에 저장)

| 헤더 | 의미 |
|---|---|
| `retry-after` | 재시도까지 초(모든 429 에 있진 않음 — 지출 한도 429 에는 없음) |
| `anthropic-ratelimit-requests-limit / -remaining / -reset` | RPM 한도 / 잔여 / 리셋(RFC3339) |
| `anthropic-ratelimit-tokens-limit / -remaining / -reset` | 현재 가장 제한적인 토큰 한도(천 단위 반올림) |
| `anthropic-ratelimit-input-tokens-limit / -remaining / -reset` | ITPM (cache_read 토큰 미포함) |
| `anthropic-ratelimit-output-tokens-limit / -remaining / -reset` | OTPM |
| `anthropic-priority-input-tokens-*`, `anthropic-priority-output-tokens-*` | Priority Tier 전용 |
| `request-id` | 모든 응답. SDK 객체 `._request_id`, 지원 문의 시 사용 |

- 출처: https://platform.claude.com/docs/en/api/errors.md · https://platform.claude.com/docs/en/api/rate-limits.md · https://platform.claude.com/docs/en/cli-sdks-libraries/sdks/python.md

In [ ]:
h = adapter.health()
if LIVE:
    print("health ok:", h.ok, "| status:", h.status_code, "| latency_ms:", h.latency_ms, "| via:", h.checked_via)
    print("ratelimit headers:", h.ratelimit)
    if h.error:
        print("error:", to_dict(h.error))
else:
    print("[BLOCKED: no key]", "| health.ok =", h.ok, "| error.category =", h.error.category)

In [ ]:
# (a) 클라이언트 TypeError: SDK 1.x 는 temperature 파라미터 자체가 없다 → 네트워크 이전에 실패 (키 불필요, 오프라인 재현)
from apicompare.providers.anthropic_adapter import error_from_exception

offline_client = anthropic.Anthropic(api_key="INVALID-KEY-FOR-TEST", max_retries=0, timeout=10)   # 요청은 보내지 않는다
try:
    offline_client.messages.create(model=MODEL, max_tokens=8, messages=[{"role": "user", "content": "ping"}], temperature=0.5)
    print("(a) 예상과 다르게 통과 — SDK 버전 확인")
except TypeError as exc:
    err_a = error_from_exception(exc)
    print("(a) TypeError →", err_a.category, "|", err_a.message[:160])
except Exception as exc:
    err_a = error_from_exception(exc)
    print("(a) 다른 예외:", type(exc).__name__, err_a.category)

In [ ]:
# (b) 서버 400: extra_body 로 temperature 를 억지로 보내면 현행 모델이 invalid_request_error 로 거부 (유료 호출 아님 — 400 은 과금 없음)
if LIVE:
    try:
        adapter.client.messages.create(model=MODEL, max_tokens=8, messages=[{"role": "user", "content": "ping"}], extra_body={"temperature": 0.5})
        print("(b) 예상과 다르게 200 — 모델이 temperature 를 허용함(구모델?)")
        err_b = None
    except anthropic.APIStatusError as exc:     # BadRequestError 도 여기로 들어온다
        err_b = error_from_exception(exc)
        print("(b) HTTP", exc.status_code, exc.type, "→", err_b.category, "|", err_b.message[:160])
    except Exception as exc:
        err_b = error_from_exception(exc)
        print("(b) 다른 예외:", type(exc).__name__, err_b.category)
else:
    err_b = None
    print("[BLOCKED: no key]")

In [ ]:
# 의도적 오류 재현 (LIVE 전용): 잘못된 키(401) / 없는 모델(404) / 초단기 timeout. 모두 CallResult.error 로 돌아온다(예외 없음).
errors = {"client_typeerror": to_dict(err_a), "server_400_temperature": to_dict(err_b)}
if LIVE:
    bad = get_adapter("anthropic", api_key="INVALID-KEY-FOR-TEST", timeout_s=30)
    errors["bad_key"] = to_dict(bad.generate("ping", model=MODEL, max_output_tokens=8).error)
    errors["bad_model"] = to_dict(adapter.generate("ping", model="model-does-not-exist-123", max_output_tokens=8).error)
    quick = get_adapter("anthropic", timeout_s=0.001)
    errors["timeout"] = to_dict(quick.generate("ping", model=MODEL, max_output_tokens=8).error)
    for k, v in errors.items():
        print(f"{k:24} → {v['category'] if v else None} status={v['status_code'] if v else None} type={v['provider_type'] if v else None}")
else:
    errors["note"] = "키가 없어 401/404/timeout 재현은 생략 (client_typeerror 는 오프라인 재현)"
    print("[BLOCKED: no key]", "| client_typeerror:", err_a.category)

관찰 메모: (실행 후 채우기) — health latency, ratelimit remaining 값, (b) 의 실제 400 메시지 원문, timeout 시 exception_class.

In [ ]:
payload = {"health": to_dict(h), "intentional_errors": errors}
if h.ok:
    print("saved:", save_evidence("anthropic", "06_errors", payload, "live", model=MODEL))
else:
    note = "no key" if not LIVE else (h.error.category if h.error else "health failed")
    print("saved:", save_evidence("anthropic", "06_errors", payload, "blocked", note=note, model=MODEL))
    print(f"[BLOCKED: {note}]")

## 7. 호출 예제

- 어댑터 없이 SDK 를 직접 쓰는 최소 예제(공식 Quickstart 기반, 10줄 이하) + 동일 요청의 curl. 키는 환경변수 `ANTHROPIC_API_KEY` 로만(SDK 가 자동으로 읽음).
- 설치: `pip install anthropic` (Python 3.10+, 1.x). 스트리밍은 `with client.messages.stream(...) as s: for t in s.text_stream: ...; s.get_final_message()`.
- 출처: https://platform.claude.com/docs/en/get-started.md · https://platform.claude.com/docs/en/cli-sdks-libraries/sdks/python.md

In [ ]:
if LIVE:
    import anthropic
    client = anthropic.Anthropic(max_retries=0, timeout=30)          # ANTHROPIC_API_KEY 환경변수 사용
    msg = client.messages.create(model=MODEL, max_tokens=64,
        messages=[{"role": "user", "content": PROMPT}])
    print("".join(b.text for b in msg.content if b.type == "text"))
    print(msg.stop_reason, "| in:", msg.usage.input_tokens, "out:", msg.usage.output_tokens, "| request_id:", msg._request_id)
else:
    msg = None
    print("[BLOCKED: no key]")

In [ ]:
# 동일 요청의 curl (키는 $ANTHROPIC_API_KEY 표기 — 값 출력 금지)
CURL = f'''curl https://api.anthropic.com/v1/messages \\
  -H "x-api-key: $ANTHROPIC_API_KEY" \\
  -H "anthropic-version: 2023-06-01" \\
  -H "content-type: application/json" \\
  -d '{{"model": "{MODEL}", "max_tokens": 64, "messages": [{{"role": "user", "content": "{PROMPT}"}}]}}' '''
print(CURL)
if msg is not None:
    print("\n응답 원본 필드:", sorted(msg.to_dict(mode="json").keys()))
else:
    print("\n[BLOCKED: no key]")

관찰 메모: (실행 후 채우기) — 예제 응답과 §2 어댑터 결과의 usage 가 비슷한지, curl 로도 같은 응답 구조가 오는지.

In [ ]:
if msg is not None:
    payload = {"python": "client.messages.create(model, max_tokens=64, messages=[...]) → ''.join(text blocks)", "curl": CURL,
               "response": {"id": msg.id, "model": msg.model, "stop_reason": msg.stop_reason, "usage": msg.usage.to_dict(mode="json")}}
    print("saved:", save_evidence("anthropic", "07_example", payload, "live", model=MODEL))
else:
    blocked("07_example")

## 8. 남긴 evidence

`evidence/anthropic/*.json` 7개가 매트릭스 셀의 근거다. 배지: `[LIVE]` 실측 / `[DOCS]` 문서 / `[OBSERVED]` 관찰 / `[BLOCKED: 사유]` 키·권한 없음.
키 없이 실행했다면 7개 모두 `[BLOCKED: no key]`(§4 는 `no admin key`) 가 정상이다. 키를 넣고 다시 실행하면 §1·§6 은 크레딧 없이도 LIVE 가 된다.

In [ ]:
rows = summary("anthropic")
print(f"{'item':14} {'항목':14} {'badge':28} saved_at")
for r in rows:
    print(f"{r['item']:14} {r['item_ko']:14} {r['badge']:28} {r['saved_at']}")